In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re

In [88]:
data = pd.read_csv('data/online_retail_II.csv')
data.shape

(1067371, 8)

### Business question: - Refine Later ###
* “How does price impact demand, and what price maximizes revenue?”
*  What is the optimal price point?
*  How sensitive are customers to price changes?
*  Do promotions actually increase revenue or just volume?

### Project Outline - Refine Later ###
1. EDA
    * Price distributions
    * Demand distibution (p vs. Q)
    * Promo comperitors
2. DGP
    * Endogeniety? 
    * Simultaneity? 
    * Promo treatment effects? 
3. Cleaning/Preprocessing
4. Feature Eng
    * Demand Signals
        * Rolling Ave
    * Price in context
    * Market in context
5. Elasticity Estimation
    * log-OLS
6. Addressing Endogeneity in Pricing
    * FE estimation
    * IV estimation
7. Validation
8. RevOpt
    * Simulate Rev/Profit curves
    * Identify Max and regionality within P/Q
9. A/B testing promos
    * OlS
    * DiD
10. Elasticity by segment
11. Demand & Revenue Curve Visualization
12. Streamlit? 


### Data Description ###
* InvoiceNo: Invoice number. Nominal. A 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter 'c', it indicates a cancellation.
* StockCode: Product (item) code. Nominal. A 5-digit integral number uniquely assigned to each distinct product.
* Description: Product (item) name. Nominal.
* Quantity: The quantities of each product (item) per transaction. Numeric.
* InvoiceDate: Invoice date and time. Numeric. The day and time when a transaction was generated.
* Price: Unit price. Numeric. Product price per unit in sterling (Â£).
* CustomerID: Customer number. Nominal. A 5-digit integral number uniquely assigned to each customer.
* Country: Country name. Nominal. The name of the country where a customer resides.

In [89]:
data['Refund'] = np.where(data['Price'] < 0, 1, 0)
data['Return'] = np.where(data['Quantity'] < 0, 1, 0 )
returned_sales = data[(data['Return'] == 1) | (data['Refund'] == 1)]


In [90]:
returned_sales['Description'].value_counts(ascending=False).head(10)

Description
Manual                                537
REGENCY CAKESTAND 3 TIER              347
POSTAGE                               229
BAKING SET 9 PIECE RETROSPOT          211
STRAWBERRY CERAMIC TRINKET BOX        184
Discount                              172
WHITE HANGING HEART T-LIGHT HOLDER    135
check                                 123
WHITE CHERRY LIGHTS                   121
RED RETROSPOT CAKE STAND              111
Name: count, dtype: int64

In [91]:
sales_data = data[(data['Refund'] == 0) & (data['Return'] == 0)]
sales_data.shape

(1044416, 10)

In [92]:
sales_data['Clean_Desc'] = (
    sales_data['Description']
    .fillna("")
    .str.upper()
    .str.replace(r"[^A-Z\s]", " ", regex = True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [93]:
top_desc = sales_data['Clean_Desc'].value_counts(ascending= False).head(25)
print("Top full descriptions:")
print(top_desc)


Top full descriptions:
Clean_Desc
WHITE HANGING HEART T LIGHT HOLDER    5783
REGENCY CAKESTAND TIER                4065
JUMBO BAG RED RETROSPOT               3395
ASSORTED COLOUR BIRD ORNAMENT         2939
PARTY BUNTING                         2742
LUNCH BAG BLACK SKULL                 2484
LUNCH BAG SUKI DESIGN                 2473
STRAWBERRY CERAMIC TRINKET BOX        2429
JUMBO STORAGE BAG SUKI                2400
FRENCH BLUE METAL DOOR SIGN           2303
HEART OF WICKER SMALL                 2299
JUMBO SHOPPER VINTAGE RED PAISLEY     2275
TEATIME FAIRY CAKE CASES              2257
PAPER CHAIN KIT S CHRISTMAS           2198
REX CASH CARRY JUMBO SHOPPER          2190
LUNCH BAG SPACEBOY DESIGN             2186
HOME BUILDING BLOCK WORD              2172
WOODEN FRAME ANTIQUE WHITE            2151
LUNCH BAG CARS BLUE                   2141
NATURAL SLATE HEART CHALKBOARD        2119
HEART OF WICKER LARGE                 2090
WOODEN PICTURE FRAME WHITE FINISH     2085
PACK OF PINK PAISLEY

In [113]:
def get_words (df, col):
    all_words = (
        df[col]
        .str.split()
        .explode()
    )

    stopwords = {
     "THE", "AND", "OF", "IN", "TO", "FOR", "ON", "WITH", "A", "AN",
     "SET", "PACK", "SMALL", "LARGE", "ASSORTED"
    }

    word_counts = (
        all_words[
            all_words.notna()
            & (all_words.str.len() > 2)
            & (~all_words.isin(stopwords))
        ]
        .value_counts()
        .head(50)
    )
    print("\nTop single words:")
    print(word_counts)
get_words(sales_data, 'Clean_Desc')


Top single words:
Clean_Desc
BAG           91439
RED           91005
HEART         78132
PINK          64048
RETROSPOT     57376
VINTAGE       54861
DESIGN        53286
WHITE         49966
BOX           49755
METAL         45041
CAKE          44892
CHRISTMAS     44073
BLUE          41588
HANGING       36187
LIGHT         35746
SIGN          35226
JUMBO         34759
HOLDER        34241
PAPER         30608
LUNCH         29986
GLASS         26645
TEA           26316
CARD          25569
DECORATION    24780
WOODEN        23468
CASES         23185
BOTTLE        22943
SPOTTY        22226
HOT           21839
WATER         21171
ROSE          20527
CERAMIC       20244
SPACEBOY      18267
MUG           18060
PAISLEY       17931
FAIRY         17247
BLACK         16985
TIN           16923
POLKADOT      16711
CREAM         16614
HOME          16495
GREEN         16177
SPOT          16144
PARTY         15366
GARDEN        15183
FELTCRAFT     15076
MINI          15036
LOVE          14496
BUNTING   

In [115]:
stopwords = {
     "THE", "AND", "OF", "IN", "TO", "FOR", "ON", "WITH", "A", "AN",
     "SET", "PACK", "SMALL", "LARGE", "ASSORTED"
    }

def get_bigrams(text: str) -> list[str]:
    words = [w for w in text.split() if len(w) > 2 and w not in stopwords]
    return [" ".join(pair) for pair in zip(words, words[1:])]

bigram_counter = Counter()

for desc in sales_data["Clean_Desc"]:
    bigram_counter.update(get_bigrams(desc))

top_bigrams = pd.Series(dict(bigram_counter)).sort_values(ascending=False).head(50)

print("\nTop bigrams:")
print(top_bigrams)


Top bigrams:
RED RETROSPOT        29615
METAL SIGN           27937
JUMBO BAG            26504
LIGHT HOLDER         23464
LUNCH BAG            21811
HOT WATER            19696
WATER BOTTLE         19696
CAKE CASES           18963
RED SPOTTY           14808
HANGING HEART        13033
FAIRY CAKE           11392
CHARLOTTE BAG        10570
DOLLY GIRL            9858
UNION JACK            9143
VINTAGE CHRISTMAS     8671
HEART LIGHT           8620
LUNCH BOX             8175
CAKE STAND            8051
BAG PINK              7976
DRAWER KNOB           7595
BAG RED               7520
BAG VINTAGE           7486
PINK POLKADOT         7448
PLASTERS TIN          7424
BIRTHDAY CARD         7258
RETRO SPOT            7187
HAND WARMER           7110
TRINKET BOX           6918
HEART DECORATION      6793
BAG SUKI              6672
SWEET HOME            6483
HOME SWEET            6472
PAPER CHAIN           6442
CHAIN KIT             6442
RIBBON REEL           6131
ALARM CLOCK           6101
ENGLISH ROSE  

### Need to clean categories - other under 20% ###

In [202]:
categories = {
    "Furniture": r"\b(TABLE|CHAIR|BENCH|SEAT|CABINET|BED|DESK|STOOL|SOFA|DRAWER|RACK)\b",
    "Kitchen": r"\b(PAN|FOOD|PANTRY|BREAD|CUTLERY|MUG|CUP|TEACUP|JAM MAKING|BOWL|PLATE|SPOON|SPOONS|LADLE|FORK|KNIFE|GLASS|TEAPOT|TRAY|BOTTLE|LUNCH|DINNER|CAKE|CAKES|KITCHEN|CAKESTAND|TEA|COOKIE|SNACK)\b",
    "Garden": r"\b(GARDEN|GARDENING|GARDENERS|PARASOL|WATERING CAN)\b",
    "Decor": r"\b(CANDLE|FLAG|FRAME|LAMP|VASE|ORNAMENT|DECORATION|WREATH|CLOCK|SIGN|HANGER|HOOK|HANGING|HOLDER|LIGHT|LIGHTS|MAT|DOORMAT|HOME|HOUSE)\b",
    "Storage": r"\b(BOX|BASKET|TIN|JAR|CONTAINER|BAG|CASE|RETROSPOT|WICKER|JUG)\b",
    "Textiles": r"\b(CUSHION|BLANKET|PILLOW|RUG|CURTAIN|TOWEL|NAPKIN|RIBBON|RIBBONS)\b",
    "Toys": r"\b(TOY|DOLL|DOLLY|BLOCK|CARDS|PLAY|PLAYHOUSE|JIGSAW|PUZZLE|GAME)\b",
    "Papers": r"\b(CARD|PAPER|NOTEBOOK|JOURNAL|BOOK|TISSUES|GIFT TAGS|POSTAGE|LUGGAGE TAG)\b",
    "Crafts": r"\b(ART|PENCIL|PEN|PAINT|MARKER|DRAWING|SEWING|CRAFT|FELTCRAFT|PENCILS|CHALK|CHALKBOARD)\b",
    "Clothes": r"\b(HAT|SHIRT|SHOES|BRACELET|APRON|HAND WARMER|SHOPPER)\b",
    "Holiday": r"\b(CHRISTMAS|BIRTHDAY|PARTY|THANKSGVING|VALENTINES|GIFT)\b"
}
sales_data["Category"] = "Other"

for category, pattern in categories.items():
    mask = sales_data["Clean_Desc"].str.contains(pattern, na=False, regex=True)
    sales_data.loc[mask, "Category"] = category

print("\nCategory counts:")
print(sales_data["Category"].value_counts())

C:\Users\Kolby.Porter\AppData\Local\Temp\ipykernel_15440\2434676497.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = sales_data["Clean_Desc"].str.contains(pattern, na=False, regex=True)
C:\Users\Kolby.Porter\AppData\Local\Temp\ipykernel_15440\2434676497.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = sales_data["Clean_Desc"].str.contains(pattern, na=False, regex=True)
C:\Users\Kolby.Porter\AppData\Local\Temp\ipykernel_15440\2434676497.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = sales_data["Clean_Desc"].str.contains(pattern, na=False, regex=True)
C:\Users\Kolby.Porter\AppData\Local\Temp\ipykernel_15440\2434676497.py:17: UserWarning: This pattern is interpreted as a regular expression, a


Category counts:
Category
Other        196695
Storage      192614
Decor        187323
Kitchen      151389
Holiday       79342
Papers        65438
Crafts        56042
Toys          33180
Clothes       25797
Textiles      21293
Garden        18059
Furniture     17244
Name: count, dtype: int64


In [203]:
bigram_counter2 = Counter()

other_sales = sales_data[sales_data['Category'] == 'Other']

for desc in other_sales["Clean_Desc"]:
    bigram_counter2.update(get_bigrams(desc))

top_bigrams = pd.Series(dict(bigram_counter2)).sort_values(ascending=False).head(50)

print("\nTop bigrams:")
print(top_bigrams)


Top bigrams:
UNION JACK          3791
KEY FOB             2766
VINTAGE UNION       2512
WHITE HEART         2143
JACK BUNTING        2117
ENGLISH ROSE        2049
IVORY TRELLIS       1919
HEART IVORY         1919
BATH SPONGE         1767
TRINKET POT         1708
HEART FILIGREE      1700
FILIGREE DOVE       1700
RED SPOTTY          1658
CHARLIE LOLA        1653
RED WHITE           1571
ANT WHITE           1564
CREAM SWEETHEART    1427
WASHING GLOVES      1411
KEY RING            1405
PAINTED METAL       1318
HEART SHAPED        1269
RED WOOLLY          1265
HOTTIE WHITE        1265
WOOLLY HOTTIE       1265
FRIDGE MAGNETS      1211
BLACK BOARD         1210
BOARD ANT           1210
WOOD BLACK          1210
WHITE FINISH        1210
MINI CASES          1207
KNICK KNACK         1199
KNACK TINS          1199
SPACEBOY DESIGN     1184
BLING KEY           1181
TOAST ITS           1181
BLUE POLKADOT       1179
CRYSTAL HEART       1175
LETTER BLING        1170
SPOTTY BUNTING      1160
PASSPORT CO

In [204]:
get_words(other_sales, 'Clean_Desc')



Top single words:
Clean_Desc
HEART          19176
RED            11864
PINK           11700
WHITE          11542
VINTAGE        10152
BLUE            8077
DESIGN          7699
WOODEN          6926
ROSE            6576
WRAP            6129
TRADITIONAL     4842
BUNTING         4800
BLACK           4763
MINI            4704
CREAM           4601
METAL           4430
FLOWER          4290
KEY             4192
UNION           4189
MIRROR          3856
MAGNETS         3826
WOOD            3807
JACK            3791
GARLAND         3688
SPOTTY          3625
TINS            3602
CANDLES         3516
PAINTED         3487
DOORSTOP        3267
IVORY           3265
RING            3176
RETRO           3043
UMBRELLA        3030
BUTTERFLY       3022
SPOT            2966
STICKERS        2837
PURSE           2822
LOVE            2814
FOB             2766
GREEN           2711
GINGHAM         2560
SPACEBOY        2550
POLKADOT        2484
BALLOONS        2407
CASES           2378
BOARD           2338
FELT

In [205]:
heart_sales = sales_data[(sales_data['Category'] == "Other")&(sales_data['Clean_Desc'].str.contains("PANTRY", na = False))]

bigram_counter3 = Counter()

for desc in heart_sales["Clean_Desc"]:
    bigram_counter3.update(get_bigrams(desc))

top_bigrams = pd.Series(dict(bigram_counter3)).sort_values(ascending=False).head(50)

#print("\nTop bigrams:")
#print(top_bigrams)
print(heart_sales['Clean_Desc'].value_counts(ascending = False).head(10))

Series([], Name: count, dtype: int64)
